In [ ]:
# Parameters — edit these before running
WORKSPACE_ID  = "990dbc7b-f5d1-4bc8-a929-9dfd509a5d52"  # Your Fabric workspace ID
LAKEHOUSE_ID  = "ec851642-fa89-42bc-aebf-2742845d36fe"  # Your lakehouse ID
SCALE_SIZE    = "small"   # small | medium | large | fabric_demo
SEED          = 42


In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.retail import RetailDomain
from azure.identity import InteractiveBrowserCredential
from deltalake import write_deltalake

SOUND_BI_TENANT = "2536810f-20e1-4911-a453-4409fd96db8a"
cred = InteractiveBrowserCredential(tenant_id=SOUND_BI_TENANT)
token = cred.get_token("https://storage.azure.com/.default").token
print(f"Auth OK — token len={len(token)}")


In [ ]:
spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale=SCALE_SIZE, seed=SEED)
print(f"Generated {sum(len(df) for df in result.tables.values()):,} rows across {len(result.tables)} tables")
for table, df in result.tables.items():
    print(f"  {table}: {len(df):,} rows")


In [ ]:
errors = result.verify_integrity()
assert errors == [], f"FK integrity errors: {errors}"
print("FK integrity: PASS")


In [ ]:
storage_opts = {"bearer_token": token, "use_emulator": "false"}

for table_name, df in result.tables.items():
    path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/retail_{table_name}"
    write_deltalake(path, df, mode="overwrite", storage_options=storage_opts, schema_mode="overwrite")
    print(f"  Written: retail_{table_name} ({len(df):,} rows)")

print("Seeding write: DONE")


## Streaming Mode
Streams each table as it generates — write table N while table N+1 is still being built.

In [ ]:
for table_name, df in spindle.generate_stream(domain=RetailDomain(), scale=SCALE_SIZE, seed=SEED):
    path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/retail_stream_{table_name}"
    write_deltalake(path, df, mode="overwrite", storage_options=storage_opts, schema_mode="overwrite")
    print(f"  Streamed: retail_stream_{table_name} ({len(df):,} rows)")

print("Streaming write: DONE")


In [ ]:
from deltalake import DeltaTable

print("Validation — reading back from OneLake:")
for table_name in result.tables:
    path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/retail_{table_name}"
    dt = DeltaTable(path, storage_options=storage_opts)
    count = len(dt.to_pandas())
    expected = len(result.tables[table_name])
    status = "OK" if count == expected else "MISMATCH"
    print(f"  {status} retail_{table_name}: {count:,} rows (expected {expected:,})")
